# Notebook 01 — Data Exploration

**Project:** Wafer Defect Detection & Yield Risk Dashboard  
**Phase:** Phase 1 — Dataset Understanding  

## Goals

1. Load the WM-811K dataset
2. Inspect the dataset structure (columns, data types, shapes)
3. Understand how wafer maps are stored
4. Extract and analyze defect class labels
5. Visualize sample wafer maps per defect class
6. Analyze class distribution
7. Identify dataset challenges (imbalance, unlabeled data, variable sizes)

## Why This Step Matters

Before training any model, we must understand the data.
Assumptions made at this stage will affect every downstream decision:
preprocessing strategy, class imbalance handling, model input shape, and evaluation approach.

## 1. Imports and Setup

In [ ]:
import sys
import types
import importlib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# Paths
DATA_RAW = Path('../data/raw')
OUTPUT_DIR = Path('../outputs/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_RAW / 'LSWMD.pkl'

print(f'Dataset path: {DATASET_PATH}')
print(f'Dataset exists: {DATASET_PATH.exists()}')

## 2. Load the Dataset

The WM-811K dataset was pickled with an old version of pandas (Python 2 era).  
Modern pandas (2.x+) cannot load it directly — `pd.read_pickle()` raises `ModuleNotFoundError: No module named 'pandas.indexes'`.

**Fix:** Load directly with `pickle.load(..., encoding='latin1')` after patching `sys.modules`  
to remap old pandas namespace paths. See `wiki/errors-and-fixes.md` for the full explanation.

In [ ]:
# Patch sys.modules so old pandas.indexes namespace resolves correctly
_REMAP = {
    'pandas.indexes':            'pandas.core.indexes',
    'pandas.indexes.base':       'pandas.core.indexes.base',
    'pandas.indexes.numeric':    'pandas.core.indexes.numeric',
    'pandas.indexes.range':      'pandas.core.indexes.range',
    'pandas.indexes.frozen':     'pandas.core.indexes.frozen',
    'pandas.indexes.api':        'pandas.core.indexes.api',
    'pandas.indexes.category':   'pandas.core.indexes.category',
    'pandas.indexes.datetimes':  'pandas.core.indexes.datetimes',
    'pandas.indexes.multi':      'pandas.core.indexes.multi',
}
for _old, _new in _REMAP.items():
    if _old not in sys.modules:
        try:
            sys.modules[_old] = importlib.import_module(_new)
        except ImportError:
            sys.modules[_old] = types.ModuleType(_old)

print('Loading dataset (this may take ~30 seconds for a 2 GB file)...')
with open(DATASET_PATH, 'rb') as f:
    df = pickle.load(f, encoding='latin1')

print(f'Dataset loaded. Shape: {df.shape}')

## 3. Inspect Dataset Structure

In [ ]:
print('=== Columns ===')
print(df.columns.tolist())

print('\n=== Shape ===')
print(df.shape)

print('\n=== First 3 rows (excluding waferMap column for readability) ===')
df.drop(columns=['waferMap']).head(3)

In [ ]:
# Inspect a single wafer map
sample = df.iloc[100]
wmap = np.array(sample['waferMap'])
print(f'waferMap shape: {wmap.shape}')
print(f'waferMap unique values: {np.unique(wmap).tolist()}')
print(f'  0 = background, 1 = passing die, 2 = failing die')
print(f'\nfailureType raw value: {sample["failureType"]}')

## 4. Extract and Clean Labels

The `failureType` column stores labels as nested lists: `[['Center']]` or `[['']]` for unlabeled.

**Note:** The column `trianTestLabel` contains a typo ("trian" not "train") — this is in the original dataset.

In [ ]:
def extract_label(failure_type):
    try:
        label = failure_type[0][0]
        return 'unlabeled' if (label == '' or label is None) else label
    except (IndexError, TypeError):
        return 'unlabeled'

df['label'] = df['failureType'].apply(extract_label)

print('Unique labels found:')
print(df['label'].unique())

## 5. Class Distribution Analysis

In [ ]:
total = len(df)
labeled = df[df['label'] != 'unlabeled']
unlabeled_count = total - len(labeled)

print(f'Total samples:     {total:>8,}')
print(f'Labeled:           {len(labeled):>8,}  ({len(labeled)/total*100:.1f}%)')
print(f'Unlabeled:         {unlabeled_count:>8,}  ({unlabeled_count/total*100:.1f}%)')

In [ ]:
labeled_counts = labeled['label'].value_counts()

fig, ax = plt.subplots(figsize=(11, 5))
colors = plt.cm.tab10.colors
bars = ax.bar(labeled_counts.index, labeled_counts.values, color=colors[:len(labeled_counts)], edgecolor='white')
ax.set_title('Defect Class Distribution — WM-811K (Labeled Samples Only)', fontsize=13, fontweight='bold')
ax.set_xlabel('Defect Class', fontsize=11)
ax.set_ylabel('Number of Samples', fontsize=11)
ax.tick_params(axis='x', rotation=25)

for bar, cnt in zip(bars, labeled_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{cnt:,}', ha='center', va='bottom', fontsize=8.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=150)
plt.show()
print('Saved: outputs/figures/class_distribution.png')

## 6. Wafer Map Size Distribution

In [ ]:
df['map_rows'] = df['waferMap'].apply(lambda x: np.array(x).shape[0])
df['map_cols'] = df['waferMap'].apply(lambda x: np.array(x).shape[1])

print(f'Row sizes — min: {df["map_rows"].min()}  max: {df["map_rows"].max()}  '
      f'mean: {df["map_rows"].mean():.1f}  unique: {df["map_rows"].nunique()}')
print(f'Col sizes — min: {df["map_cols"].min()}  max: {df["map_cols"].max()}  '
      f'mean: {df["map_cols"].mean():.1f}  unique: {df["map_cols"].nunique()}')
print('\nConclusion: variable sizes — all maps must be resized to a fixed shape before training.')

## 7. Visualize Sample Wafer Maps Per Class

Wafer map encoding: `0` = background, `1` = passing die, `2` = failing die

In [ ]:
DEFECT_CLASSES = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring',
                  'Loc', 'Near-full', 'Random', 'Scratch', 'none']

wafer_cmap = ListedColormap(['black', '#4CAF50', '#F44336'])

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Sample Wafer Maps by Defect Class — WM-811K', fontsize=15, fontweight='bold')

for ax, cls in zip(axes.flat, DEFECT_CLASSES):
    subset = labeled[labeled['label'] == cls]
    if len(subset) == 0:
        ax.set_title(f'{cls}\n(no samples)', fontsize=10)
        ax.axis('off')
        continue
    wmap = np.array(subset.iloc[0]['waferMap'])
    ax.imshow(wmap, cmap=wafer_cmap, vmin=0, vmax=2, interpolation='nearest')
    ax.set_title(f'{cls}\n(n={len(subset):,})', fontsize=10)
    ax.axis('off')

legend_els = [
    mpatches.Patch(color='black', label='Background'),
    mpatches.Patch(color='#4CAF50', label='Pass die'),
    mpatches.Patch(color='#F44336', label='Fail die'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=3, fontsize=11, frameon=True)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.savefig(OUTPUT_DIR / 'sample_wafer_maps.png', dpi=150)
plt.show()
print('Saved: outputs/figures/sample_wafer_maps.png')

## 8. Class Imbalance Summary

In [ ]:
class_stats = labeled_counts.reset_index()
class_stats.columns = ['Class', 'Count']
class_stats['Percentage'] = (class_stats['Count'] / class_stats['Count'].sum() * 100).round(1)
class_stats['ImbalanceRatio'] = (class_stats['Count'].max() / class_stats['Count']).round(0).astype(int)

print(class_stats.to_string(index=False))
print(f'\nImbalance ratio (none vs Near-full): {class_stats["ImbalanceRatio"].max()}x')

## 9. Summary of Findings

| Property | Value |
|---|---|
| Total samples | 811,457 |
| Labeled samples | 172,950 (21.3%) |
| Unlabeled samples | 638,507 (78.7%) |
| Defect classes | 9 |
| Dominant class | `none` — 147,431 samples (85.2%) |
| Rarest class | `Near-full` — 149 samples (0.1%) |
| Imbalance ratio | **989x** |
| Wafer map row range | 6 – 300 (129 unique sizes) |
| Wafer map col range | 3 – 205 (120 unique sizes) |

**Key decisions from EDA:**
- Use only labeled samples (172,950) for supervised training
- Resize all wafer maps to **64×64** using nearest-neighbor interpolation
- Use **class weights in CrossEntropyLoss** to address 989x imbalance
- Use **stratified train/val/test split** (70/15/15)
- Never use accuracy alone — report **per-class F1-score**

**Next step:** `notebooks/02_preprocessing.ipynb`